# Build retriever indexes (Google Colab)

This notebook **only builds** the FinQA and TAT-QA retriever indexes and downloads the bundles. **No evaluation** — no Claude API calls, no cost.

1. Clone the repo and install dependencies.
2. Download FinQA and TAT-QA datasets.
3. (Optional) Generate `first_5_rows.json` for FinQA.
3b. **Overlap check:** FinQA train vs test overlap (ensures test is out-of-sample; run before building test index).
4. Build **FinQA train** retriever index (GPU or CPU) from `train_qa.json`.
4a. **Download FinQA train index** — do this if you only want the train index, then you can disconnect.
4a2. Build **FinQA test** retriever index from `test.json` (same pipeline as 4; for `eval_runner.py --split test`).
4a3. **Download FinQA test index** — unzip into `data/rag/FinQA/test/`.
4b. Build **TAT-QA** retriever index (GPU or CPU) — from **all splits** (train, dev, test). You can run only sections **2, 4b, 4c** to rebuild just the TAT-QA index.
4c. **Download TAT-QA index** — right after building TAT-QA.
6. **Local:** Verify indexes load (section 6).
7. **Colab only:** Disconnect and delete runtime (section 7) to release the T4 GPU and avoid extra usage.

Run every cell in order on Colab; then unzip the downloaded bundles locally for use with **demo_agentic_rag_eval.ipynb** (evaluation and display). To **rerun only TAT-QA indexing**: run section 2 (download datasets), then 4b and 4c.


## Purpose

This notebook is for **building and downloading** the retriever indexes only. Use **demo_agentic_rag_eval.ipynb** to run RAG evaluation (FinQA + TAT-QA) and display results — that notebook assumes indexes are already built or downloaded.


## 1. Clone repo and install dependencies

The first code cell below also registers a **global exception handler**: if any later cell raises an error, the Colab runtime will disconnect automatically so you don't keep incurring GPU usage after a failure.

**Optional:** In Colab, add a Secret named **`HUGGINGFACE_HUB_TOKEN`** (your HF token from https://huggingface.co/settings/tokens). The cell will set it so FinQA and TAT-QA downloads and model loads use authenticated requests (higher rate limits, no "unauthenticated requests" warning).


In [ ]:
REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
REPO_DIR = "ocr-agentic-rag"

import os
os.environ["COLAB_GPU"] = os.environ.get("COLAB_GPU", "1")

# Hugging Face token from Colab Secret (HUGGINGFACE_HUB_TOKEN) for higher rate limits and faster downloads (FinQA + TAT-QA).
try:
    from google.colab import userdata
    _hf_token = userdata.get("HUGGINGFACE_HUB_TOKEN")
    os.environ["HUGGINGFACE_HUB_TOKEN"] = _hf_token
    os.environ["HF_TOKEN"] = _hf_token
    print("Hugging Face token set from Colab secret HUGGINGFACE_HUB_TOKEN.")
except Exception:
    pass  # not in Colab or secret not set

if os.path.isdir(REPO_DIR):
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull")
else:
    get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)

!pip install -q sentence-transformers faiss-cpu rank_bm25 llama-index-core anthropic langgraph pandas

# On Colab: if any cell raises an exception, disconnect runtime to stop GPU billing.
try:
    from google.colab import runtime
    def _disconnect_on_error(etype, evalue, tb):
        try:
            runtime.unassign()
        except Exception:
            pass
        return False  # let IPython show the traceback
    get_ipython().set_custom_exc((BaseException,), _disconnect_on_error)
except Exception:
    pass  # not in Colab or API changed


## 2. Download RAG datasets (FinQA + TAT-QA)

This step uses `scripts/download_rag_datasets.py` to fetch the FinQA and TAT-QA datasets into the locations expected by the evaluation pipeline and dataset adapters.

For example, to download only FinQA you could run:

```bash
!python scripts/download_rag_datasets.py --datasets finqa
```

In the cell below I download **both** FinQA and TAT-QA.


In [ ]:
!python scripts/download_rag_datasets.py --datasets finqa tatqa


## 3. Generate first_5_rows for FinQA (optional)

This is a small convenience helper that writes out the first 5 rows of the FinQA training set to `data/rag/FinQA/train/first_5_rows.json` so you can quickly inspect the schema and example questions.


In [ ]:
import json
from pathlib import Path

train_dir = Path("data/rag/FinQA/train")
train_qa_path = train_dir / "train_qa.json"
first_5_path = train_dir / "first_5_rows.json"

if train_qa_path.exists():
    with open(train_qa_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    rows = data[:5] if isinstance(data, list) else []
    with open(first_5_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Wrote {first_5_path} ({len(rows)} rows).")
else:
    print("train_qa.json not found; run section 2 first.")


## 4. Full overlap report: out-of-sample QA overlap (report only)

This section **reports** overlaps; it does **not** modify the index or chunking. Overlapping samples are **excluded at eval time** in `eval_dataset_adapters.py`. When I exclude a QA from eval, that sample is never run or scored — so the **overlap corpus** (that QA / that document for that question) is effectively removed from the eval set.

**Single reference:** Reference = train_qa.json + tatqa_dataset_test_gold.json. Both eval splits checked against this combined set; no QA from -eval" splits appears in the -reference" .

1. **FinQA test** (`test.json`): No test QA may appear in the reference (train_qa **or** tatqa_test_gold). Excluding overlapping QAs = those samples and their corpus are not evaluated.
2. **TAT-QA dev** (`tatqa_dataset_dev.json`): No dev QA may appear in the reference (train_qa **or** tatqa_test_gold). Excluding overlapping QAs = those samples and their corpus are not evaluated.

Run this before building the test index (section 7) to confirm out-of-sample integrity. Overlaps found here are already excluded in the adapters.

In [ ]:
import json
import re
from pathlib import Path

def _normalize(s: str) -> str:
    """Normalize for overlap check: lower, strip, collapse whitespace."""
    if s is None:
        return ""
    return " ".join(re.split(r"\s+", str(s).strip().lower()))

def _load_finqa_qa_list(path: Path) -> list:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    return data if isinstance(data, list) else []

def _load_tatqa(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else []

def check_finqa_train_test_overlap(train_path: Path, test_path: Path):
    """Check that no test QA pair (or question) repeats in train. Returns (overlap_qa, overlap_q_only, stats)."""
    train_data = _load_finqa_qa_list(train_path)
    test_data = _load_finqa_qa_list(test_path) if test_path.exists() else []
    train_qa = set()
    train_q = set()
    for entry in train_data:
        qa = entry.get("qa") or {}
        q = _normalize((qa if isinstance(qa, dict) else {}).get("question") or entry.get("question") or "")
        ans = qa.get("exe_ans") or qa.get("answer") or entry.get("answer") or ""
        a = _normalize(str(ans))
        if q:
            train_q.add(q)
        if q and a:
            train_qa.add((q, a))
    overlap_qa = []
    overlap_q_only = []
    for idx, entry in enumerate(test_data):
        qa = entry.get("qa") or {}
        q = _normalize((qa if isinstance(qa, dict) else {}).get("question") or entry.get("question") or "")
        ans = qa.get("exe_ans") or qa.get("answer") or entry.get("answer") or ""
        a = _normalize(str(ans))
        eid = entry.get("id") or entry.get("filename") or str(idx)
        if (q, a) in train_qa:
            overlap_qa.append({"id": eid, "question": (entry.get("qa") or {}).get("question") or entry.get("question"), "answer": ans})
        elif q in train_q:
            overlap_q_only.append({"id": eid, "question": (entry.get("qa") or {}).get("question") or entry.get("question"), "answer": ans})
    stats = {
        "train_entries": len(train_data),
        "train_unique_qa": len(train_qa),
        "train_unique_q": len(train_q),
        "test_entries": len(test_data),
        "test_in_train_qa": len(overlap_qa),
        "test_q_in_train_only": len(overlap_q_only),
    }
    return overlap_qa, overlap_q_only, stats

def _load_tatqa(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else []

def iter_tatqa_qa(data: list):
    for doc in data:
        for q in doc.get("questions", []):
            ans = q.get("answer", [])
            a = ans[0] if isinstance(ans, list) and ans else ans
            yield q.get("uid"), _normalize(q.get("question") or ""), _normalize(str(a) if a is not None else "")

def check_tatqa_test_gold_vs_dev(test_gold_path: Path, dev_path: Path):
    """Reference = test_gold; eval = dev. Find dev samples that appear in test_gold."""
    ref = _load_tatqa(test_gold_path)
    eval_data = _load_tatqa(dev_path)
    ref_qa = set()
    ref_q = set()
    for uid, q, a in iter_tatqa_qa(ref):
        if q:
            ref_q.add(q)
        if q and a:
            ref_qa.add((q, a))
    overlap_qa = []
    overlap_q = []
    for uid, q, a in iter_tatqa_qa(eval_data):
        if (q, a) in ref_qa:
            overlap_qa.append(uid)
        elif q in ref_q:
            overlap_q.append(uid)
    stats = {
        "ref_samples": sum(len(d.get("questions", [])) for d in ref),
        "eval_samples": sum(len(d.get("questions", [])) for d in eval_data),
        "ref_unique_qa": len(ref_qa),
        "ref_unique_q": len(ref_q),
        "eval_in_ref_qa": len(overlap_qa),
        "eval_q_in_ref_only": len(overlap_q),
    }
    return overlap_qa, overlap_q, stats

repo_root = Path(".").resolve()
train_path = repo_root / "data" / "rag" / "FinQA" / "train" / "train_qa.json"
test_gold_path = repo_root / "data" / "rag" / "TAT-QA" / "tatqa_dataset_test_gold.json"

# Build COMBINED reference: train_qa.json + tatqa_dataset_test_gold.json (no eval QA may appear in either)
ref_qa = set()
ref_q = set()
train_data = _load_finqa_qa_list(train_path)
for entry in train_data:
    qa = entry.get("qa") or {}
    q = _normalize((qa if isinstance(qa, dict) else {}).get("question") or entry.get("question") or "")
    a = _normalize(str(qa.get("exe_ans") or qa.get("answer") or entry.get("answer") or ""))
    if q:
        ref_q.add(q)
    if q and a:
        ref_qa.add((q, a))
for doc in _load_tatqa(test_gold_path):
    for q in doc.get("questions", []):
        ans = q.get("answer", [])
        a = ans[0] if isinstance(ans, list) and ans else ans
        qn = _normalize(q.get("question") or "")
        an = _normalize(str(a) if a is not None else "")
        if qn:
            ref_q.add(qn)
        if qn and an:
            ref_qa.add((qn, an))
print("Combined reference (train_qa + tatqa_dataset_test_gold): unique (Q,A):", len(ref_qa), "unique Q:", len(ref_q))
print()

# --- 1. FinQA test (test.json) vs combined reference ---
print("=" * 60)
print("1. FinQA test (test.json) vs combined reference")
print("=" * 60)
test_path = repo_root / "data" / "rag" / "FinQA" / "test" / "test.json"
if not test_path.exists():
    print("  test.json not found — add from https://github.com/czyssrs/FinQA (dataset/test.json).")
else:
    test_data = _load_finqa_qa_list(test_path)
    finqa_exclude = []
    for idx, entry in enumerate(test_data):
        qa = entry.get("qa") or {}
        q = _normalize((qa if isinstance(qa, dict) else {}).get("question") or entry.get("question") or "")
        a = _normalize(str(qa.get("exe_ans") or qa.get("answer") or entry.get("answer") or ""))
        sid = entry.get("id") or f"{entry.get('filename', f'doc_{idx}')}-{idx}"
        if (q, a) in ref_qa or q in ref_q:
            finqa_exclude.append(sid)
    print(f"  Test entries: {len(test_data)}")
    print(f"  Test samples in reference (excluded at eval): {len(finqa_exclude)}")
    if finqa_exclude:
        print("  Sample IDs:", sorted(finqa_exclude)[:10], "..." if len(finqa_exclude) > 10 else "")
        print("  Excluded in FinQAAdapter.FINQA_TEST_EXCLUDE_SAMPLE_IDS → overlap corpus not evaluated.")
    else:
        print("  OK: test is out-of-sample.")

# --- 2. TAT-QA dev (tatqa_dataset_dev.json) vs combined reference ---
print()
print("=" * 60)
print("2. TAT-QA dev (tatqa_dataset_dev.json) vs combined reference")
print("=" * 60)
dev_path = repo_root / "data" / "rag" / "TAT-QA" / "tatqa_dataset_dev.json"
if not dev_path.exists():
    print("  tatqa_dataset_dev.json not found.")
else:
    dev_data = _load_tatqa(dev_path)
    tatqa_exclude = []
    for doc in dev_data:
        for q in doc.get("questions", []):
            uid = q.get("uid")
            ans = q.get("answer", [])
            a = ans[0] if isinstance(ans, list) and ans else ans
            qn = _normalize(q.get("question") or "")
            an = _normalize(str(a) if a is not None else "")
            if (qn, an) in ref_qa or qn in ref_q:
                tatqa_exclude.append(uid)
    n_dev = sum(len(d.get("questions", [])) for d in dev_data)
    print(f"  Dev samples: {n_dev}")
    print(f"  Dev samples in reference (excluded at eval): {len(tatqa_exclude)}")
    if tatqa_exclude:
        print("  Sample UIDs:", sorted(tatqa_exclude)[:10], "..." if len(tatqa_exclude) > 10 else "")
        print("  Excluded in TATQAAdapter.TATQA_DEV_EXCLUDE_SAMPLE_IDS → overlap corpus not evaluated.")
    else:
        print("  OK: dev is out-of-sample.")

print()
print("Done. Excluding overlapping QAs at eval time removes those samples (and their corpus) from the eval set.")

## 5. Build FinQA **train** retriever index (GPU on Colab)

I now build the **FinQA train** retriever index over `data/rag/FinQA/train/train_qa.json` using the `HybridRetriever`. **All pre-index steps run automatically before embedding** (so they are done on Colab before the index is built):

- **Section tagging:** Chunks are labeled with document section (income statement, balance sheet, notes) from context.
- **Unit parsing:** Units (millions, thousands, per-share, quarterly) are detected at chunk ingestion and stored in metadata.
- **Deduplication:** Content is hashed; duplicates are merged (first kept, `duplicate_count` set); index is built over unique chunks.
- **Provenance:** Page (from FinQA id), section, and table ID are stored per chunk for retrieval preference.

- On Colab, I default to **GPU** (`COLAB_GPU=1`). For local or CPU-only runs, use the `--cpu` flag.
- **Table-aware (Level 3):** Set `TABLE_AWARE = True` below for one chunk per table row with headers inline. After building, run **section 6** to download and replace your local `data/rag/FinQA/train/finqa_retriever_index/`.
- **No dedup:** Set `NO_DEDUP = True` below to skip content-hash dedup so every corpus_id keeps its own table chunks (avoids missing chunks when multiple entries share the same table). Index will be larger.

The script `scripts/build_finqa_embeddings_colab.py` chunks the corpus, runs the pre-index pipeline above, then builds and saves the index bundle.

**One Colab run:** With `TABLE_AWARE = True` and `NO_DEDUP = True` (default below), a single build secures **header-per-row table-aware chunking**, **section tagging**, **unit parsing** (from chunk + doc context), **no dedup** (each corpus_id keeps its chunks), and **provenance**. See **data/proof/INDEX_CHECKLIST.md** for a verification checklist before you run.


In [ ]:
import os

# Table-aware (Level 3): one chunk per table row with column headers inline. Recommended to avoid mislabeled values (RAG_LESSONS.md). Re-download index (section 6) and replace local bundle after building.
TABLE_AWARE = True
# Skip dedup so every corpus_id keeps its own table chunks (avoids missing chunks when multiple entries share the same table, e.g. page_64.pdf-1..4). Index will be larger.
NO_DEDUP = True

batch_size = os.environ.get("EMBED_BATCH_SIZE", "170")
use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
table_aware_flag = " --table_aware" if TABLE_AWARE else ""
no_dedup_flag = " --no_dedup" if NO_DEDUP else ""
if use_gpu:
    get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --batch_size {batch_size}{table_aware_flag}{no_dedup_flag}")
else:
    get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --cpu{table_aware_flag}{no_dedup_flag}")


## 6. Download FinQA **train** retriever index

After building the **FinQA train** index (section 5), you can **download or save the bundle here** and then disconnect (e.g. run section 12). You don't have to build TAT-QA. Set **SAVE_TO_DRIVE** and **DRIVE_FOLDER** below.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

SAVE_TO_DRIVE = False  # True = copy to your Google Drive (same as drive.google.com)
DRIVE_FOLDER = "colab_index_bundles"

def _zip_and_save(name: str, src_dir: Path, zip_name: str):
    if not src_dir.exists():
        print(f"{name}: index not found at {src_dir}; skip.")
        return
    zip_path = Path(zip_name)
    shutil.make_archive(zip_path.with_suffix(""), "zip", src_dir.parent, src_dir.name)
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount failed:", e)
            files.download(str(zip_path))
            return
        drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_name)
        print(f"{name}: saved to Drive: {drive_dir / zip_name}")
    else:
        files.download(str(zip_path))
        print(f"{name}: download triggered. Unzip into data/rag/FinQA/train/ so that finqa_retriever_index/ exists.")

_zip_and_save("FinQA", Path("data/rag/FinQA/train/finqa_retriever_index"), "finqa_retriever_index.zip")


## 7. Build FinQA **test** retriever index

**Prerequisite:** `data/rag/FinQA/test/test.json` must exist. Section 2 does **not** download it; add it manually from [FinQA dataset](https://github.com/czyssrs/FinQA/tree/main/dataset).

Builds a **separate** retriever index from `data/rag/FinQA/test/test.json` for evaluation with `--split test`. Uses the same pipeline as section 5 (table-aware chunking, section tagging, unit parsing, optional no-dedup). Output: `data/rag/FinQA/test/finqa_retriever_index/`. Run this after the overlap report (section 4) and optionally after building the train index (section 5). Use the same `TABLE_AWARE` and `NO_DEDUP` as in section 5 for consistency.

In [ ]:
import os
from pathlib import Path

# Use same TABLE_AWARE and NO_DEDUP as section 5 (set there). If you only run this block, define them:
try:
    TABLE_AWARE
except NameError:
    TABLE_AWARE = True
try:
    NO_DEDUP
except NameError:
    NO_DEDUP = True

test_json = Path("data/rag/FinQA/test/test.json")
test_index_dir = Path("data/rag/FinQA/test/finqa_retriever_index")
if not test_json.exists():
    print("test.json not found at", test_json, "- add it from https://github.com/czyssrs/FinQA (dataset/test.json) and re-run section 2 or download manually.")
else:
    batch_size = os.environ.get("EMBED_BATCH_SIZE", "170")
    use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
    table_aware_flag = " --table_aware" if TABLE_AWARE else ""
    no_dedup_flag = " --no_dedup" if NO_DEDUP else ""
    if use_gpu:
        get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --train_qa {test_json} --output {test_index_dir} --batch_size {batch_size}{table_aware_flag}{no_dedup_flag}")
    else:
        get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --train_qa {test_json} --output {test_index_dir} --cpu{table_aware_flag}{no_dedup_flag}")
    print("FinQA test index saved to", test_index_dir)

## 8. Download FinQA **test** index

After building the **FinQA test** index (section 7), download or save the bundle here. Unzip into `data/rag/FinQA/test/` so that `finqa_retriever_index/` exists for `eval_runner.py --split test`.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

# Same options as section 6 (Download FinQA train index)
SAVE_TO_DRIVE = False
DRIVE_FOLDER = "colab_index_bundles"

def _zip_and_save(name: str, src_dir: Path, zip_name: str):
    if not src_dir.exists():
        print(f"{name}: index not found at {src_dir}; skip.")
        return
    zip_path = Path(zip_name)
    shutil.make_archive(zip_path.with_suffix(""), "zip", src_dir.parent, src_dir.name)
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount failed:", e)
            files.download(str(zip_path))
            return
        drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_name)
        print(f"{name}: saved to Drive: {drive_dir / zip_name}")
    else:
        files.download(str(zip_path))
        print(f"{name}: download triggered. Unzip into data/rag/FinQA/test/ so that finqa_retriever_index/ exists.")

_zip_and_save("FinQA test", Path("data/rag/FinQA/test/finqa_retriever_index"), "finqa_test_retriever_index.zip")

## 9. Build TAT-QA retriever index (GPU on Colab)

The TAT-QA index is built from **all splits** (train, dev, test): `tatqa_dataset_train.json`, `tatqa_dataset_dev.json`, and `tatqa_dataset_test_gold.json`. Including train and dev ensures the full corpus is in the index and avoids retrieval gaps; evaluation still uses the test split only.

**Rerun only TAT-QA:** If you already have the repo and datasets (section 2), you can run **sections 9 and 10** alone to rebuild and download just the TAT-QA index.

Mirrors section 5 for **TAT-QA**: the same **pre-index steps** (header-per-row chunking with `--table_aware`, section tagging, unit parsing, dedup, provenance) run in one go. Set `TABLE_AWARE = True` (same variable as in section 5) for Level 3 row-level serialization. Output: `data/rag/TAT-QA/tatqa_retriever_index/`. After building, run **section 10** to download and replace your local TAT-QA index. Verification: **data/proof/INDEX_CHECKLIST.md**.

In [ ]:
import os

# Use same TABLE_AWARE as section 5 (set there). If you only ran section 5, define it here: TABLE_AWARE = True
try:
    TABLE_AWARE
except NameError:
    TABLE_AWARE = True

batch_size = os.environ.get("EMBED_BATCH_SIZE", "250")
use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
table_aware_flag = " --table_aware" if TABLE_AWARE else ""
if use_gpu:
    get_ipython().system(f"python scripts/build_tatqa_embeddings_colab.py --output data/rag/TAT-QA/tatqa_retriever_index --batch_size {batch_size}{table_aware_flag}")
else:
    get_ipython().system(f"python scripts/build_tatqa_embeddings_colab.py --output data/rag/TAT-QA/tatqa_retriever_index --cpu{table_aware_flag}")

## 10. Download TAT-QA index

After building TAT-QA above (section 9), you can **download or save the TAT-QA index here**. Set **SAVE_TO_DRIVE** and **DRIVE_FOLDER** below (same as in section 6).


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

SAVE_TO_DRIVE = False  # True = copy to your Google Drive (same as drive.google.com)
DRIVE_FOLDER = "colab_index_bundles"

def _zip_and_save(name: str, src_dir: Path, zip_name: str):
    if not src_dir.exists():
        print(f"{name}: index not found at {src_dir}; skip.")
        return
    zip_path = Path(zip_name)
    shutil.make_archive(zip_path.with_suffix(""), "zip", src_dir.parent, src_dir.name)
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount failed:", e)
            files.download(str(zip_path))
            return
        drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_name)
        print(f"{name}: saved to Drive: {drive_dir / zip_name}")
    else:
        files.download(str(zip_path))
        print(f"{name}: download triggered. Unzip into data/rag/TAT-QA/ so that tatqa_retriever_index/ exists.")

_zip_and_save("TAT-QA", Path("data/rag/TAT-QA/tatqa_retriever_index"), "tatqa_retriever_index.zip")


**Tip — Edge stuck on download permission:** To allow Colab downloads in Microsoft Edge so they don't hang: (1) Open **edge://settings/downloads** and turn off "Ask where to save each file before downloading" if you're okay with a default folder, or (2) In **edge://settings/content** → Automatic downloads, add `https://colab.research.google.com` to allowed sites. Alternatively use **SAVE_TO_DRIVE = True** in sections 4a/4c and skip browser download entirely.


## 11. Local test: verify retriever indexes

Run this **on your local PC** after you have placed the index bundles under `data/rag/FinQA/train/`, `data/rag/FinQA/test/` (if using test split), and/or `data/rag/TAT-QA/`. It loads each available index and runs a quick retrieval to confirm the bundle works (no GPU required for this test).

In [ ]:
from pathlib import Path
import sys

# Run from repo root so data/ paths resolve
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rag_system.retrieval import HybridRetriever

def test_index(name: str, index_dir: Path, query: str, corpus_id: str = None):
    if not index_dir.exists() or not (index_dir / "meta.json").exists():
        print(f"{name}: index not found at {index_dir} (skip).")
        return
    print(f"{name}: loading from {index_dir} ...")
    retriever = HybridRetriever()
    retriever.load_index_bundle(str(index_dir))
    results = retriever.retrieve(query, top_k=2, corpus_id=corpus_id)
    print(f"  Query: {query[:60]}...")
    print(f"  Retrieved {len(results)} chunk(s).")
    if results:
        text = results[0][0].text
        print(f"  First chunk preview: {text[:150]}...")
    print()

# FinQA: expect index at data/rag/FinQA/train/finqa_retriever_index
test_index(
    "FinQA",
    repo_root / "data" / "rag" / "FinQA" / "train" / "finqa_retriever_index",
    "What was the interest expense in 2009?",
    corpus_id=None,  # or a specific doc id from your data to test corpus scoping
)

# TAT-QA: expect index at data/rag/TAT-QA/tatqa_retriever_index
test_index(
    "TAT-QA",
    repo_root / "data" / "rag" / "TAT-QA" / "tatqa_retriever_index",
    "What is the total sales amount?",
    corpus_id=None,
)

# FinQA test: expect index at data/rag/FinQA/test/finqa_retriever_index (built in section 7)
test_index(
    "FinQA test",
    repo_root / "data" / "rag" / "FinQA" / "test" / "finqa_retriever_index",
    "What is the net change in net revenue during 2015 for entergy corporation?",
    corpus_id=None,
)

print("Done. If the indexes you use were found and retrieved chunks, you can run eval_runner locally (e.g. --split train or --split test for FinQA).")

## 12. Disconnect and delete runtime (Colab only)

Run this cell **on Colab** after you have downloaded the index bundles (sections 6, 8, 10). It disconnects and deletes the runtime so the T4 GPU is released and you stop incurring usage—helps stay within free-tier limits. If you run this notebook locally, skip this cell (it will no-op or remind you to disconnect manually in Colab).

In [ ]:
# Release the Colab runtime (disconnect + delete) to stop GPU usage.
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("Could not unassign runtime:", e)
    print("In Colab: use Runtime → Disconnect and delete runtime to stop GPU usage.")